# LeetCode 809 — Expressive Words bằng Python

Link đề: https://leetcode.com/problems/expressive-words/ · Độ khó: **Medium** · Chủ đề: String, Two Pointers (hai con trỏ), Array

Skeleton LeetCode:

```python
class Solution:
    def expressiveWords(self, s: str, words: List[str]) -> int:
        ...
```

Ghi chú: bài xử lý chuỗi theo **nhóm ký tự liên tiếp giống nhau** (run). Khi viết "heeellooo" ta thấy các nhóm `h`, `eee`, `ll`, `ooo`. Một từ truy vấn "kéo dài được" thành `s` nếu **từng nhóm khớp** theo luật "chỉ được kéo dài nhóm lên **từ 3 ký tự trở lên**". Notebook đi từ việc mô phỏng đúng định nghĩa (sinh mọi chuỗi kéo dài), tới mã hóa nhóm (run-length), hai con trỏ, biểu thức chính quy, và gom các từ theo "khung chữ cái".

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: một phép **kéo dài** chọn một nhóm ký tự `c` rồi thêm các `c` sao cho nhóm có **độ dài >= 3**; đếm số từ trong `words` có thể trở thành `s` sau một số phép kéo dài (kể cả 0 phép).
2. Nắm các công cụ Python cần thiết: `itertools.groupby` để tách nhóm, mã hóa **run-length** (danh sách `(ký tự, độ dài)`), hai con trỏ trên chuỗi, `re.fullmatch` với lượng từ `{n,}`, `collections.defaultdict`.
3. Rút ra luật so khớp từng cặp nhóm `(c1 trong s, c2 trong t)`: **cùng ký tự**, và `c1 == c2` hoặc (`c1 >= 3` và `c1 > c2`).
4. Đi qua 5 cách: sinh mọi chuỗi kéo dài (vét cạn theo định nghĩa), so sánh danh sách run-length, hai con trỏ không tạo danh sách, biểu thức chính quy, tiền xử lý một lần cho `s` rồi nhóm theo khung chữ cái; đối chiếu bằng oracle vét cạn.
5. Tránh các lỗi kinh điển: bỏ luật "từ 3 trở lên", áp luật `>= 3` lên **nhóm của từ truy vấn** thay vì nhóm của `s`, quên kiểm tra hai chuỗi cùng hết, đếm ký tự bỏ qua thứ tự, so sánh độ dài thay vì từng nhóm.

## 2. Đề bài & ví dụ

Đôi khi người ta lặp chữ để nhấn mạnh: `"hello" -> "heeellooo"`, `"hi" -> "hiiii"`. Trong chuỗi như `"heeellooo"` ta có các **nhóm** ký tự liên tiếp giống nhau: `"h"`, `"eee"`, `"ll"`, `"ooo"`.

Cho chuỗi `s` và mảng các chuỗi truy vấn `words`. Một từ truy vấn là **stretchy** (kéo dài được) nếu có thể biến nó thành `s` bằng **một số phép kéo dài** (có thể 0 phép), trong đó **phép kéo dài** là: chọn một nhóm gồm ký tự `c`, rồi thêm một số ký tự `c` vào nhóm sao cho **kích thước nhóm >= 3**.

Ví dụ: từ `"hello"` ta có thể kéo dài nhóm `"o"` thành `"hellooo"`, nhưng **không** thể được `"helloo"` vì nhóm `"oo"` chỉ có 2 ký tự (< 3). Cũng có thể kéo `"ll"` thành `"lllll"` để được `"helllllooo"`. Với `s = "helllllooo"` từ `"hello"` là stretchy.

Trả về **số từ** trong `words` là stretchy.

**Ví dụ 1**

- Input: `s = "heeellooo"`, `words = ["hello", "hi", "helo"]`
- Output: `1`
- Giải thích: kéo `"e"` và `"o"` của `"hello"` được `"heeellooo"`. `"helo"` không được vì nhóm `"ll"` (kích thước 2) không thể sinh từ nhóm `"l"` (một phép kéo dài chỉ cho nhóm >= 3).

**Ví dụ 2**

- Input: `s = "zzzzzyyyyy"`, `words = ["zzyy", "zy", "zyy"]`
- Output: `3`

**Ràng buộc:**

- `1 <= s.length, words.length <= 100`
- `1 <= words[i].length <= 100`
- `s` và `words[i]` chỉ gồm chữ cái thường.

In [1]:
from typing import List, Tuple
from itertools import groupby
from collections import deque, defaultdict
import re, random, time

examples = [
    ("heeellooo", ["hello", "hi", "helo"], 1),
    ("zzzzzyyyyy", ["zzyy", "zy", "zyy"], 3),
]
for s, words, expected in examples:
    print(f"s={s!r} words={words} kỳ vọng={expected}")

s='heeellooo' words=['hello', 'hi', 'helo'] kỳ vọng=1
s='zzzzzyyyyy' words=['zzyy', 'zy', 'zyy'] kỳ vọng=3


## 3. Kiến thức nền

### 3.1. Nhóm ký tự liên tiếp (run) và `itertools.groupby`

`groupby(iterable)` gom các phần tử **liên tiếp bằng nhau**. Với chuỗi `"heeellooo"` ta được các nhóm `h`, `eee`, `ll`, `ooo`. Chú ý `groupby` chỉ gom phần tử **liền kề**: chuỗi `"aba"` cho 3 nhóm (không phải 2).

In [2]:
for ch, grp in groupby("heeellooo"):
    print(repr(ch), list(grp))
print([(ch, len(list(g))) for ch, g in groupby("aba")])

'h' ['h']
'e' ['e', 'e', 'e']
'l' ['l', 'l']
'o' ['o', 'o', 'o']
[('a', 1), ('b', 1), ('a', 1)]


### 3.2. Mã hóa run-length (RLE)

**Run-length encoding** biểu diễn chuỗi bằng danh sách cặp `(ký tự, độ dài nhóm)`: `"heeellooo" -> [('h',1), ('e',3), ('l',2), ('o',3)]`. So sánh hai chuỗi theo nhóm trở nên dễ: so ký tự và so độ dài từng nhóm.

In [3]:
def rle(s: str) -> List[Tuple[str, int]]:
    return [(ch, len(list(g))) for ch, g in groupby(s)]

for w in ["heeellooo", "hello", "helo", "hi", "zzzzzyyyyy", "a"]:
    print(f"{w!r:14} -> {rle(w)}")

'heeellooo'    -> [('h', 1), ('e', 3), ('l', 2), ('o', 3)]
'hello'        -> [('h', 1), ('e', 1), ('l', 2), ('o', 1)]
'helo'         -> [('h', 1), ('e', 1), ('l', 1), ('o', 1)]
'hi'           -> [('h', 1), ('i', 1)]
'zzzzzyyyyy'   -> [('z', 5), ('y', 5)]
'a'            -> [('a', 1)]


### 3.3. Hai con trỏ trên hai chuỗi

Ta có thể duyệt `s` và `t` đồng thời bằng hai chỉ số `i`, `j`, mỗi lần nhảy qua **cả một nhóm** (đếm độ dài nhóm bằng vòng `while`), thay vì tạo danh sách. Điều kiện kết thúc: cả hai chỉ số cùng chạm cuối.

In [4]:
def run_length_at(s: str, i: int) -> int:
    """Độ dài nhóm bắt đầu tại chỉ số i."""
    k = i
    while k < len(s) and s[k] == s[i]:
        k += 1
    return k - i

s = "heeellooo"
i = 0
while i < len(s):
    L = run_length_at(s, i)
    print(f"nhóm {s[i]!r} bắt đầu tại {i}, dài {L}")
    i += L

nhóm 'h' bắt đầu tại 0, dài 1
nhóm 'e' bắt đầu tại 1, dài 3
nhóm 'l' bắt đầu tại 4, dài 2
nhóm 'o' bắt đầu tại 6, dài 3


### 3.4. Biểu thức chính quy: lượng từ `{n}` và `{n,}`

`re.fullmatch(pattern, text)` khớp **toàn bộ** `text`. Lượng từ `c{n}` nghĩa là đúng `n` ký tự `c`, `c{n,}` nghĩa là **ít nhất** `n` ký tự `c`, `(?:A|B)` là "A hoặc B" (nhóm không đánh số).

In [5]:
print(bool(re.fullmatch(r"he{3,}ll{2}o{3,}", "heeellooo")))
print(bool(re.fullmatch(r"he{3,}ll{2}o{3,}", "heellooo")))
print(bool(re.fullmatch(r"(?:l{2}|l{3,})", "lll")), bool(re.fullmatch(r"(?:l{2}|l{3,})", "ll")), bool(re.fullmatch(r"(?:l{2}|l{3,})", "l")))

False
False
True True False


### 3.5. `defaultdict` và gom nhóm theo khóa

`defaultdict(list)` tự tạo list rỗng cho khóa mới: tiện để gom các từ theo "khung chữ cái" (chuỗi các ký tự của nhóm, bỏ độ dài). Ví dụ khung của `"heeellooo"` là `"helo"`.

In [6]:
by_skeleton = defaultdict(list)
for w in ["hello", "hi", "helo", "heello", "hhhi"]:
    skeleton = "".join(ch for ch, _ in rle(w))
    by_skeleton[skeleton].append(w)
print(dict(by_skeleton))

{'helo': ['hello', 'helo', 'heello'], 'hi': ['hi', 'hhhi']}


## 4. Ý tưởng nền tảng / trực giác (tính tay)

Phép kéo dài **chỉ làm dài thêm một nhóm**, không đổi ký tự và không thêm/xóa nhóm. Vậy `t` stretchy thành `s` thì **dãy các ký tự nhóm phải giống hệt nhau** (cùng "khung"), và với **từng nhóm** (ký tự `c`, `n_t` trong `t`, `n_s` trong `s`):

- Nếu `n_s == n_t`: không cần làm gì (hợp lệ).
- Nếu `n_s > n_t`: phải kéo dài nhóm, mà kéo dài chỉ cho nhóm **>= 3** nên cần `n_s >= 3`.
- Nếu `n_s < n_t`: không thể thu nhỏ -> sai.

Ví dụ 1: `s = "heeellooo"` -> `[(h,1), (e,3), (l,2), (o,3)]`.

| từ | RLE | từng nhóm | kết quả |
|---|---|---|---|
| `hello` | `(h,1) (e,1) (l,2) (o,1)` | `h`: 1=1 ✓; `e`: 3>1 và 3>=3 ✓; `l`: 2=2 ✓; `o`: 3>1 và 3>=3 ✓ | có |
| `hi` | `(h,1) (i,1)` | khung `hi` khác `helo` | không |
| `helo` | `(h,1) (e,1) (l,1) (o,1)` | `l`: 2>1 nhưng 2<3 ✗ | không |

Đáp án: 1.

In [7]:
def explain(s, t):
    rs, rt = rle(s), rle(t)
    print(f"s={s!r} {rs}\n  t={t!r} {rt}")
    if [c for c, _ in rs] != [c for c, _ in rt]:
        print("  -> khung ký tự khác nhau: KHÔNG")
        return False
    for (c, ns), (_, nt) in zip(rs, rt):
        ok = ns == nt or (ns > nt and ns >= 3)
        print(f"  nhóm {c!r}: n_s={ns}, n_t={nt} -> {'ok' if ok else 'SAI'}")
        if not ok:
            return False
    print("  -> CÓ")
    return True

for w in ["hello", "hi", "helo"]:
    explain("heeellooo", w)

s='heeellooo' [('h', 1), ('e', 3), ('l', 2), ('o', 3)]
  t='hello' [('h', 1), ('e', 1), ('l', 2), ('o', 1)]
  nhóm 'h': n_s=1, n_t=1 -> ok
  nhóm 'e': n_s=3, n_t=1 -> ok
  nhóm 'l': n_s=2, n_t=2 -> ok
  nhóm 'o': n_s=3, n_t=1 -> ok
  -> CÓ
s='heeellooo' [('h', 1), ('e', 3), ('l', 2), ('o', 3)]
  t='hi' [('h', 1), ('i', 1)]
  -> khung ký tự khác nhau: KHÔNG
s='heeellooo' [('h', 1), ('e', 3), ('l', 2), ('o', 3)]
  t='helo' [('h', 1), ('e', 1), ('l', 1), ('o', 1)]
  nhóm 'h': n_s=1, n_t=1 -> ok
  nhóm 'e': n_s=3, n_t=1 -> ok
  nhóm 'l': n_s=2, n_t=1 -> SAI


## 5. Cách 1 — Brute Force: sinh mọi chuỗi có thể kéo dài từ `t`

**Ý tưởng.** Làm đúng như đề: từ `t`, áp dụng mọi phép kéo dài có thể (chọn một nhóm, kéo lên độ dài `L >= 3` và `L` lớn hơn độ dài hiện tại) cho đến khi độ dài vượt `len(s)`. Duyệt BFS trên các chuỗi và xem `s` có xuất hiện không. Vì phép kéo dài chỉ tăng độ dài nên số chuỗi hữu hạn (chặn bởi `len(s)`).

Đây là mô phỏng trực tiếp định nghĩa nên cũng là **oracle** cho `s` ngắn.

In [8]:
def reachable(t: str, limit: int, k: int = 3) -> set:
    """Mọi chuỗi có độ dài <= limit nhận được từ t bằng các phép kéo dài (nhóm sau khi kéo >= k)."""
    seen = {t}
    q = deque([t])
    while q:
        cur = q.popleft()
        pos = 0
        for ch, n in rle(cur):
            for L in range(max(n + 1, k), limit - len(cur) + n + 1):
                new = cur[:pos] + ch * L + cur[pos + n:]
                if new not in seen:
                    seen.add(new)
                    q.append(new)
            pos += n
    return seen

def stretchy_brute(s: str, t: str) -> bool:
    return len(t) <= len(s) and s in reachable(t, len(s))

def expressive_brute(s: str, words: List[str]) -> int:
    return sum(stretchy_brute(s, t) for t in words)

for s, words, expected in examples:
    got = expressive_brute(s, words)
    print(s, words, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

heeellooo ['hello', 'hi', 'helo'] -> 1 OK
zzzzzyyyyy ['zzyy', 'zy', 'zyy'] -> 3 OK


**Độ phức tạp và nút thắt.** Số chuỗi tạo ra có thể rất lớn (mỗi nhóm có nhiều độ dài khả dĩ, tích qua các nhóm): chẳng hạn từ `"abcd"` và `limit = 40` sinh hàng chục nghìn chuỗi, chỉ để trả lời một câu hỏi có/không. Hãy đếm:

In [9]:
for lim in (8, 12, 16, 20, 24):
    print(f"t='abcd' limit={lim}: {len(reachable('abcd', lim)):>8,} chuỗi")

t='abcd' limit=8:       19 chuỗi
t='abcd' limit=12:      160 chuỗi
t='abcd' limit=16:      721 chuỗi
t='abcd' limit=20:    2,246 chuỗi
t='abcd' limit=24:    5,535 chuỗi


Nút thắt: ta **không cần sinh** các chuỗi, chỉ cần **so từng cặp nhóm** (luật ở mục 4).

## 6. Ý tưởng thuật toán / bước đột phá

Vì phép kéo dài **độc lập theo từng nhóm** và **không đổi khung ký tự**, bài toán rút gọn thành: tách `s` và `t` thành các nhóm, so khung, rồi kiểm tra điều kiện `(n_s == n_t)` hoặc `(n_s >= 3 và n_s > n_t)` cho **từng cặp nhóm**. Đó là một lần quét `O(len(s) + len(t))` cho mỗi từ.

Các cách triển khai:

| Cách | Ý tưởng |
|---|---|
| 2 | Mã hóa RLE cả hai chuỗi bằng `groupby`, so từng cặp |
| 3 | Hai con trỏ nhảy từng nhóm, không tạo danh sách |
| 4 | Biên dịch `t` thành biểu thức chính quy (`c{n,}` hoặc `(?:c{n}|c{3,})`) rồi `fullmatch` với `s` |
| 5 | Tiền xử lý RLE của `s` **một lần**; gom các từ theo khung chữ cái để chỉ so với những từ cùng khung |

## 7. Các cách tiếp theo

### 7.1. Cách 2 — So sánh danh sách run-length

**Giải thích từng đoạn.**

- `rs = rle(s)` tính một lần.
- Với mỗi `t`: `rt = rle(t)`. Nếu `len(rt) != len(rs)` -> không. Với từng cặp `(c, ns)` và `(d, nt)`: nếu `c != d` -> không; nếu `ns < nt` hoặc (`ns < 3` và `ns != nt`) -> không.
- Qua hết thì đếm.

In [10]:
def expressive_rle(s: str, words: List[str]) -> int:
    rs = rle(s)

    def ok(t: str) -> bool:
        rt = rle(t)
        if len(rt) != len(rs):
            return False
        for (c, ns), (d, nt) in zip(rs, rt):
            if c != d or ns < nt or (ns < 3 and ns != nt):
                return False
        return True

    return sum(ok(t) for t in words)

for s, words, expected in examples:
    assert expressive_rle(s, words) == expected
print("RLE OK")

RLE OK


**Code nộp LeetCode (Cách 2):**

In [11]:
class Solution:
    def expressiveWords(self, s: str, words: List[str]) -> int:
        from itertools import groupby

        def rle(x: str):
            return [(ch, len(list(g))) for ch, g in groupby(x)]

        rs = rle(s)
        ans = 0
        for t in words:
            rt = rle(t)
            if len(rt) != len(rs):
                continue
            good = True
            for (c, ns), (d, nt) in zip(rs, rt):
                if c != d or ns < nt or (ns < 3 and ns != nt):
                    good = False
                    break
            ans += good
        return ans

sol_rle = Solution().expressiveWords
print([sol_rle(s, w) for s, w, _ in examples])

[1, 3]


**Độ phức tạp (Cách 2):** thời gian `O(len(s) * 1 + sum len(t))` (mỗi chuỗi một lần quét), bộ nhớ `O(len(s) + len(t))` cho danh sách nhóm.

---

### 7.2. Cách 3 — Hai con trỏ (lời giải chuẩn)

**Giải thích từng đoạn.**

- `check(s, t)`: `i`, `j` là chỉ số trong `s`, `t`; nếu `len(t) > len(s)` trả `False` ngay.
- Lặp khi cả hai chưa hết: nếu `s[i] != t[j]` -> `False`. Đếm `c1` (độ dài nhóm trong `s` từ `i`) và `c2` (trong `t` từ `j`). Nếu `c1 < c2` hoặc (`c1 < 3` và `c1 != c2`) -> `False`. Nhảy `i += c1`, `j += c2`.
- Cuối: trả `i == len(s) and j == len(t)` (cả hai cùng hết).

In [12]:
def stretchy_two_pointers(s: str, t: str) -> bool:
    m, n = len(s), len(t)
    if n > m:
        return False
    i = j = 0
    while i < m and j < n:
        if s[i] != t[j]:
            return False
        k = i
        while k < m and s[k] == s[i]:
            k += 1
        c1 = k - i
        i, k = k, j
        while k < n and t[k] == t[j]:
            k += 1
        c2 = k - j
        j = k
        if c1 < c2 or (c1 < 3 and c1 != c2):
            return False
    return i == m and j == n

def expressive_two_pointers(s: str, words: List[str]) -> int:
    return sum(stretchy_two_pointers(s, t) for t in words)

for s, words, expected in examples:
    assert expressive_two_pointers(s, words) == expected
print("hai con trỏ OK")

hai con trỏ OK


**Code nộp LeetCode (Cách 3):**

In [13]:
class Solution:
    def expressiveWords(self, s: str, words: List[str]) -> int:
        def check(s, t):
            m, n = len(s), len(t)
            if n > m:
                return False
            i = j = 0
            while i < m and j < n:
                if s[i] != t[j]:
                    return False
                k = i
                while k < m and s[k] == s[i]:
                    k += 1
                c1 = k - i
                i, k = k, j
                while k < n and t[k] == t[j]:
                    k += 1
                c2 = k - j
                j = k
                if c1 < c2 or (c1 < 3 and c1 != c2):
                    return False
            return i == m and j == n

        return sum(check(s, t) for t in words)

sol_two = Solution().expressiveWords
print([sol_two(s, w) for s, w, _ in examples])

[1, 3]


**Độ phức tạp (Cách 3):** thời gian `O(len(s) * m + sum len(t))` với `m = len(words)` (mỗi từ quét lại `s`), bộ nhớ `O(1)`.

---

### 7.3. Cách 4 — Biên dịch `t` thành biểu thức chính quy

**Ý tưởng khác.** Với mỗi nhóm `(c, n)` của `t`, đoạn tương ứng trong `s` phải khớp:

- nếu `n >= 3`: `c{n,}` (ít nhất `n` ký tự `c`; nếu `s` có nhiều hơn thì là kéo dài, hợp lệ vì nhóm gốc đã >= 3);
- nếu `n` là 1 hoặc 2: `(?:c{n}|c{3,})` (hoặc đúng `n` ký tự, hoặc kéo lên >= 3).

Ghép các đoạn rồi `re.fullmatch(pattern, s)`. Dùng `re.escape` cho an toàn (ở đây chỉ chữ thường).

**Giải thích từng đoạn.** Mỗi `c` xuất hiện trong khung nên nhóm của `s` được "kẹp" giữa các nhóm khác ký tự; vì hai nhóm kề nhau của `t` luôn khác ký tự nên regex không bị mơ hồ.

In [14]:
def compile_word(t: str):
    parts = []
    for ch, n in rle(t):
        c = re.escape(ch)
        if n >= 3:
            parts.append(f"{c}{{{n},}}")
        else:
            parts.append(f"(?:{c}{{{n}}}|{c}{{3,}})")
    return re.compile("".join(parts))

def expressive_regex(s: str, words: List[str]) -> int:
    return sum(compile_word(t).fullmatch(s) is not None for t in words)

print(compile_word("hello").pattern)
for s, words, expected in examples:
    assert expressive_regex(s, words) == expected
print("regex OK")

(?:h{1}|h{3,})(?:e{1}|e{3,})(?:l{2}|l{3,})(?:o{1}|o{3,})
regex OK


**Code nộp LeetCode (Cách 4):**

In [15]:
class Solution:
    def expressiveWords(self, s: str, words: List[str]) -> int:
        import re
        from itertools import groupby

        ans = 0
        for t in words:
            parts = []
            for ch, g in groupby(t):
                n = len(list(g))
                if n >= 3:
                    parts.append(f"{ch}{{{n},}}")
                else:
                    parts.append(f"(?:{ch}{{{n}}}|{ch}{{3,}})")
            if re.fullmatch("".join(parts), s):
                ans += 1
        return ans

sol_regex = Solution().expressiveWords
print([sol_regex(s, w) for s, w, _ in examples])

[1, 3]


**Độ phức tạp (Cách 4):** thời gian `O(sum len(t))` để dựng mẫu cộng chi phí khớp (mẫu không có lồng lặp nên backtracking bị chặn trong thực tế, nhưng độ phức tạp tệ nhất phụ thuộc bộ máy regex); bộ nhớ `O(len(t))`.

---

### 7.4. Cách 5 — Tiền xử lý `s` một lần và gom từ theo khung chữ cái

**Ý tưởng khác.** Có tới 100 từ nhưng chỉ một `s`. Tách `s` thành `(khung, danh sách độ dài)` **một lần**. Với mỗi từ: tính khung của nó, và chỉ khi **khung bằng khung của `s`** mới so độ dài. Gom theo khung để các từ khác khung bị loại bằng một phép so sánh chuỗi (rất rẻ) thay vì duyệt nhóm.

**Giải thích từng đoạn.**

- `skeleton_s`, `counts_s` từ `s`.
- Từ `t`: dựng `skeleton_t`, `counts_t`; nếu `skeleton_t != skeleton_s` -> loại; ngược lại kiểm tra `all(ns == nt or (ns >= 3 and ns > nt))`.

In [16]:
def split_skeleton(x: str):
    runs = rle(x)
    return "".join(c for c, _ in runs), [n for _, n in runs]

def expressive_skeleton(s: str, words: List[str]) -> int:
    sk_s, cnt_s = split_skeleton(s)
    ans = 0
    for t in words:
        sk_t, cnt_t = split_skeleton(t)
        if sk_t != sk_s:
            continue
        if all(ns == nt or (ns >= 3 and ns > nt) for ns, nt in zip(cnt_s, cnt_t)):
            ans += 1
    return ans

for s, words, expected in examples:
    assert expressive_skeleton(s, words) == expected
print("khung chữ cái OK")

khung chữ cái OK


**Code nộp LeetCode (Cách 5):**

In [17]:
class Solution:
    def expressiveWords(self, s: str, words: List[str]) -> int:
        from itertools import groupby

        def split(x: str):
            keys, counts = [], []
            for ch, g in groupby(x):
                keys.append(ch)
                counts.append(len(list(g)))
            return "".join(keys), counts

        sk, cs = split(s)
        ans = 0
        for t in words:
            sk_t, ct = split(t)
            if sk_t == sk and all(a == b or (a >= 3 and a > b) for a, b in zip(cs, ct)):
                ans += 1
        return ans

sol_skeleton = Solution().expressiveWords
print([sol_skeleton(s, w) for s, w, _ in examples])

[1, 3]


**Độ phức tạp (Cách 5):** thời gian `O(len(s) + sum len(t))`, bộ nhớ `O(len(s) + max len(t))`.

## 8. TRACE từng bước

### 8.1. Cách 3 (hai con trỏ) cho `s = "heeellooo"`, `t = "hello"`

| vòng | `i` | `j` | `s[i]`, `t[j]` | `c1` | `c2` | kiểm tra | sau khi nhảy |
|---|---|---|---|---|---|---|---|
| 1 | 0 | 0 | h, h | 1 | 1 | `1 == 1` ✓ | `i = 1`, `j = 1` |
| 2 | 1 | 1 | e, e | 3 | 1 | `3 >= 1`, `3 >= 3` ✓ | `i = 4`, `j = 2` |
| 3 | 4 | 2 | l, l | 2 | 2 | `2 == 2` ✓ | `i = 6`, `j = 4` |
| 4 | 6 | 4 | o, o | 3 | 1 | ✓ | `i = 9`, `j = 5` |

Cuối: `i == 9 == len(s)` và `j == 5 == len(t)` -> `True`.

In [18]:
def trace_two_pointers(s: str, t: str) -> bool:
    m, n = len(s), len(t)
    print(f"s={s!r} (dài {m}), t={t!r} (dài {n})")
    if n > m:
        print("  t dài hơn s -> False")
        return False
    i = j = 0
    step = 0
    while i < m and j < n:
        step += 1
        if s[i] != t[j]:
            print(f"  vòng {step}: s[{i}]={s[i]!r} != t[{j}]={t[j]!r} -> False")
            return False
        k = i
        while k < m and s[k] == s[i]:
            k += 1
        c1 = k - i
        k2 = j
        while k2 < n and t[k2] == t[j]:
            k2 += 1
        c2 = k2 - j
        bad = c1 < c2 or (c1 < 3 and c1 != c2)
        print(f"  vòng {step}: nhóm {s[i]!r}: c1={c1} (trong s), c2={c2} (trong t) -> {'SAI' if bad else 'ok'}")
        if bad:
            return False
        i, j = k, k2
    res = i == m and j == n
    print(f"  kết thúc: i={i}/{m}, j={j}/{n} -> {res}")
    return res

assert trace_two_pointers("heeellooo", "hello") is True
assert trace_two_pointers("heeellooo", "helo") is False
assert trace_two_pointers("heeellooo", "hi") is False
assert trace_two_pointers("abc", "abcd") is False

s='heeellooo' (dài 9), t='hello' (dài 5)
  vòng 1: nhóm 'h': c1=1 (trong s), c2=1 (trong t) -> ok
  vòng 2: nhóm 'e': c1=3 (trong s), c2=1 (trong t) -> ok
  vòng 3: nhóm 'l': c1=2 (trong s), c2=2 (trong t) -> ok
  vòng 4: nhóm 'o': c1=3 (trong s), c2=1 (trong t) -> ok
  kết thúc: i=9/9, j=5/5 -> True
s='heeellooo' (dài 9), t='helo' (dài 4)
  vòng 1: nhóm 'h': c1=1 (trong s), c2=1 (trong t) -> ok
  vòng 2: nhóm 'e': c1=3 (trong s), c2=1 (trong t) -> ok
  vòng 3: nhóm 'l': c1=2 (trong s), c2=1 (trong t) -> SAI
s='heeellooo' (dài 9), t='hi' (dài 2)
  vòng 1: nhóm 'h': c1=1 (trong s), c2=1 (trong t) -> ok
  vòng 2: s[1]='e' != t[1]='i' -> False
s='abc' (dài 3), t='abcd' (dài 4)
  t dài hơn s -> False


### 8.2. Ví dụ 2: `s = "zzzzzyyyyy"`

`s` có hai nhóm `(z,5)`, `(y,5)`. Cả ba từ `zzyy`, `zy`, `zyy` có khung `zy`; mọi nhóm của `s` đều >= 3 và lớn hơn nhóm của từ nên cả ba hợp lệ -> 3.

In [19]:
for t in ["zzyy", "zy", "zyy"]:
    explain("zzzzzyyyyy", t)

s='zzzzzyyyyy' [('z', 5), ('y', 5)]
  t='zzyy' [('z', 2), ('y', 2)]
  nhóm 'z': n_s=5, n_t=2 -> ok
  nhóm 'y': n_s=5, n_t=2 -> ok
  -> CÓ
s='zzzzzyyyyy' [('z', 5), ('y', 5)]
  t='zy' [('z', 1), ('y', 1)]
  nhóm 'z': n_s=5, n_t=1 -> ok
  nhóm 'y': n_s=5, n_t=1 -> ok
  -> CÓ
s='zzzzzyyyyy' [('z', 5), ('y', 5)]
  t='zyy' [('z', 1), ('y', 2)]
  nhóm 'z': n_s=5, n_t=1 -> ok
  nhóm 'y': n_s=5, n_t=2 -> ok
  -> CÓ


### 8.3. Cách 4: mẫu regex và so khớp

Mẫu của `"hello"`: `h(?:e{1}|e{3,})l{2}... ` — xem `compile_word("hello").pattern` ở trên. Hãy thử khớp với `s` khác nhau:

In [20]:
pat = compile_word("hello")
print(pat.pattern)
for s in ["heeellooo", "hello", "heello", "helloo", "hhello", "hellllooo"]:
    print(f"{s!r:12} khớp? {pat.fullmatch(s) is not None}   (đối chiếu oracle: {stretchy_brute(s, 'hello')})")
    assert (pat.fullmatch(s) is not None) == stretchy_brute(s, "hello")

(?:h{1}|h{3,})(?:e{1}|e{3,})(?:l{2}|l{3,})(?:o{1}|o{3,})
'heeellooo'  khớp? True   (đối chiếu oracle: True)
'hello'      khớp? True   (đối chiếu oracle: True)
'heello'     khớp? False   (đối chiếu oracle: False)
'helloo'     khớp? False   (đối chiếu oracle: False)
'hhello'     khớp? False   (đối chiếu oracle: False)
'hellllooo'  khớp? True   (đối chiếu oracle: True)


## 9. Code nộp LeetCode

Ta chọn **Cách 3** (hai con trỏ): `O(1)` bộ nhớ phụ, bám sát cách giải thích "so từng cặp nhóm".

```python
class Solution:
    def expressiveWords(self, s: str, words: List[str]) -> int:
        def check(s, t):
            m, n = len(s), len(t)
            if n > m:
                return False
            i = j = 0
            while i < m and j < n:
                if s[i] != t[j]:
                    return False
                k = i
                while k < m and s[k] == s[i]:
                    k += 1
                c1 = k - i
                i, k = k, j
                while k < n and t[k] == t[j]:
                    k += 1
                c2 = k - j
                j = k
                if c1 < c2 or (c1 < 3 and c1 != c2):
                    return False
            return i == m and j == n

        return sum(check(s, t) for t in words)
```

In [21]:
class Solution:
    def expressiveWords(self, s: str, words: List[str]) -> int:
        def check(s, t):
            m, n = len(s), len(t)
            if n > m:
                return False
            i = j = 0
            while i < m and j < n:
                if s[i] != t[j]:
                    return False
                k = i
                while k < m and s[k] == s[i]:
                    k += 1
                c1 = k - i
                i, k = k, j
                while k < n and t[k] == t[j]:
                    k += 1
                c2 = k - j
                j = k
                if c1 < c2 or (c1 < 3 and c1 != c2):
                    return False
            return i == m and j == n

        return sum(check(s, t) for t in words)

sol = Solution()
for s, words, expected in examples:
    got = sol.expressiveWords(s, words)
    print(s, words, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

heeellooo ['hello', 'hi', 'helo'] -> 1 OK
zzzzzyyyyy ['zzyy', 'zy', 'zyy'] -> 3 OK


## 10. Vì sao đúng

**(a) Phép kéo dài bảo toàn khung.** Mỗi phép chỉ thêm ký tự `c` vào một nhóm `c`. Các nhóm kề nhau vốn khác ký tự nên sau khi kéo vẫn tách rời, và **không có nhóm nào xuất hiện/biến mất/đổi ký tự**. Vậy `t` stretchy thành `s` ⇒ hai chuỗi có cùng dãy ký tự nhóm (khung).

**(b) Điều kiện theo từng nhóm.** Các phép kéo dài trên các nhóm khác nhau **độc lập**. Với một nhóm `(c, n_t)` của `t`, các độ dài `n_s` có thể đạt được là: `n_t` (không làm gì) hoặc bất kỳ `n_s >= max(n_t + 1, 3)` (kéo một lần lên `n_s`; kéo nhiều lần cũng chỉ tạo các độ dài này vì mỗi lần kéo cho kết quả >= 3 và lớn hơn). Tức `n_s == n_t` hoặc (`n_s > n_t` và `n_s >= 3`). Phủ định: `n_s < n_t` hoặc (`n_s < 3` và `n_s != n_t`) — chính là điều kiện loại ở code.

**(c) Hết cả hai chuỗi.** Hai con trỏ nhảy theo nhóm: nếu một chuỗi còn nhóm mà chuỗi kia hết thì khung khác độ dài -> sai; kiểm tra `i == m and j == n` bắt trường hợp này.

**(d) Cách 4.** Với nhóm `(c, n)`: nếu `n >= 3` thì `n_s` hợp lệ khi `n_s >= n` (`n_s == n` hoặc `n_s > n`, đều >= 3): `c{n,}`. Nếu `n ∈ {1, 2}` thì hợp lệ khi `n_s == n` hoặc `n_s >= 3`: `(?:c{n}|c{3,})`. Nhóm kề khác ký tự nên mẫu ghép khớp từng nhóm một.

**(e) Cách 5.** Chỉ là Cách 2 với bước loại sớm theo khung.

**Kiểm chứng bằng máy:** so mọi cách với oracle BFS (Cách 1) trên **mọi** cặp chuỗi `(s, t)` trên bảng chữ cái `{a, b}` độ dài <= 7, và trên các tập từ ngẫu nhiên có seed (các từ được tạo bằng cách co các nhóm của `s` để có cả trường hợp đúng lẫn sai).

In [22]:
def all_strings(alphabet: str, max_len: int):
    out, frontier = [], [""]
    for _ in range(max_len):
        frontier = [x + c for x in frontier for c in alphabet]
        out += frontier
    return [x for x in out if x]

solutions = {
    "rle": expressive_rle,
    "two_pointers": expressive_two_pointers,
    "regex": expressive_regex,
    "skeleton": expressive_skeleton,
    "sol_rle": sol_rle,
    "sol_two": sol_two,
    "sol_regex": sol_regex,
    "sol_skeleton": sol_skeleton,
    "final": Solution().expressiveWords,
}

strings = all_strings("ab", 7)
reach_all = {t: reachable(t, 7) for t in strings}       # mọi chuỗi đạt được từ t với độ dài <= 7
checked = 0
for s in strings:
    for t in strings:
        ref = len(t) <= len(s) and s in reach_all[t]
        for name, fn in solutions.items():
            assert (fn(s, [t]) == 1) == ref, (name, s, t, ref)
        checked += 1
print("mọi cặp (s, t) trên {a,b} độ dài 1..7:", checked, "cặp x", len(solutions), "cài đặt khớp oracle BFS")

mọi cặp (s, t) trên {a,b} độ dài 1..7: 64516 cặp x 9 cài đặt khớp oracle BFS


## 11. Time / Space Complexity

Gọi `m = len(s)`, `W = len(words)`, `L = tổng độ dài các từ` (mỗi `<= 100`).

| Cách | Time | Space |
|---|---|---|
| 1. Sinh mọi chuỗi kéo dài | tệ nhất mũ theo số nhóm | rất lớn |
| 2. So danh sách RLE | `O(m + L)` | `O(m + max len)` |
| 3. Hai con trỏ | `O(m * W + L)` | `O(1)` |
| 4. Regex | `O(L)` dựng + khớp | `O(max len)` |
| 5. Khung chữ cái | `O(m + L)` | `O(m + max len)` |

Với `m, W, len <= 100` mọi cách `O(m + L)` chạy dưới 1 ms. Đo thử:

In [23]:
rng = random.Random(809)
def random_runs_string(rng, runs, alphabet="abcdefghij", maxrun=6):
    out, prev = [], ""
    for _ in range(runs):
        c = rng.choice([x for x in alphabet if x != prev])
        out.append(c * rng.randint(1, maxrun))
        prev = c
    return "".join(out)

tests = []
for _ in range(300):
    s = random_runs_string(rng, rng.randint(5, 25))
    words = []
    for _ in range(100):
        # co ngẫu nhiên các nhóm của s -> có từ đúng lẫn sai
        w = "".join(ch * max(1, min(n, rng.choice([n, n, 1, 2, 3, n + 1]))) for ch, n in rle(s))
        if rng.random() < 0.2:
            w = w[:-1] or w
        words.append(w[:100])
    tests.append((s[:100], words))

ref = None
for name in ["rle", "two_pointers", "regex", "skeleton"]:
    t0 = time.perf_counter()
    res = [solutions[name](s, words) for s, words in tests]
    dt = (time.perf_counter() - t0) * 1000
    if ref is None:
        ref = res
    assert res == ref, name
    print(f"{name:13} {len(tests)} test x 100 từ: {dt:8.1f} ms")
print("tổng số từ stretchy tìm được:", sum(ref), "/", 300 * 100)

rle           300 test x 100 từ:    115.3 ms
two_pointers  300 test x 100 từ:    192.2 ms


regex         300 test x 100 từ:   5767.7 ms
skeleton      300 test x 100 từ:    170.8 ms
tổng số từ stretchy tìm được: 18289 / 30000


## 12. Bộ test

Gồm: ví dụ của đề, edge case, phản ví dụ cho các cách sai, đối chiếu mọi cách với oracle BFS trên input nhỏ, và test ngẫu nhiên có seed (đúng ràng buộc: độ dài <= 100, chữ thường).

In [24]:
# 12.1 ví dụ đề
for s, words, expected in examples:
    assert expressive_brute(s, words) == expected
    for name, fn in solutions.items():
        assert fn(s, words) == expected, (name, s, words)
print("12.1 ví dụ đề: OK")

# 12.2 edge case (kỳ vọng viết tay, đã kiểm bằng suy luận và oracle)
edge = [
    ("a", ["a"], 1),                                # giống hệt
    ("a", ["b"], 0),                                # khác ký tự
    ("aaa", ["a"], 1),                              # kéo 'a' lên 3
    ("aa", ["a"], 0),                               # nhóm 2 < 3 không sinh được từ nhóm 1
    ("aa", ["aa"], 1),                              # nhóm 2 giữ nguyên
    ("aaa", ["aa"], 1),                             # kéo 'aa' lên 3
    ("aaaa", ["aaaa", "aaa", "aa", "a"], 4),
    ("abc", ["abc", "ab", "abcd", "aabc"], 1),
    ("abbbc", ["abc", "abbc", "abbbc"], 3),         # nhóm b: 1, 2, 3 đều kéo được lên 3
    ("abbc", ["abc", "abbc"], 1),                   # 'bb' = 2 chỉ giống chính nó
    ("aaabbb", ["ab", "aab", "abb", "aabb", "aaabb"], 5),
    ("heeellooo", ["hello", "hi", "helo", "heeellooo", "heellooo", "hellooo"], 4),   # hello, heeellooo, heellooo, hellooo
    ("a" * 100, ["a" * 100, "a", "aa", "a" * 99], 4),
    ("ab" * 50, ["ab" * 50, "ab" * 49, "a"], 1),
    ("zzzzzyyyyy", ["zzyy", "zy", "zyy"], 3),
]
for s, words, expected in edge:
    if len(s) <= 12:
        assert expressive_brute(s, words) == expected, ("oracle khác kỳ vọng", s, words)
    for name, fn in solutions.items():
        assert fn(s, words) == expected, (name, s, words, fn(s, words), expected)
print("12.2 edge case: OK", len(edge), "ca")

12.1 ví dụ đề: OK
12.2 edge case: OK 15 ca


In [25]:
# 12.3 phản ví dụ: các cách sai
def wrong_no_three(s, words):          # bỏ luật "từ 3 trở lên": chỉ cần c1 >= c2
    def ok(t):
        rs, rt = rle(s), rle(t)
        return len(rs) == len(rt) and all(a == b and ns >= nt for (a, ns), (b, nt) in zip(rs, rt))
    return sum(ok(t) for t in words)

def wrong_rule_on_t(s, words):         # áp luật >= 3 lên nhóm của t thay vì của s
    def ok(t):
        rs, rt = rle(s), rle(t)
        return len(rs) == len(rt) and all(a == b and (ns == nt or (nt >= 3 and ns > nt)) for (a, ns), (b, nt) in zip(rs, rt))
    return sum(ok(t) for t in words)

def wrong_no_end_check(s, words):      # quên kiểm tra hai chuỗi cùng hết
    def ok(t):
        m, n = len(s), len(t)
        if n > m:
            return False
        i = j = 0
        while i < m and j < n:
            if s[i] != t[j]:
                return False
            k = i
            while k < m and s[k] == s[i]:
                k += 1
            c1 = k - i
            i, k = k, j
            while k < n and t[k] == t[j]:
                k += 1
            c2 = k - j
            j = k
            if c1 < c2 or (c1 < 3 and c1 != c2):
                return False
        return True                    # SAI: thiếu i == m and j == n
    return sum(ok(t) for t in words)

def wrong_counter(s, words):           # chỉ so tần suất ký tự, bỏ qua thứ tự
    from collections import Counter
    cs = Counter(s)
    return sum(all(cs[c] >= n for c, n in Counter(t).items()) and set(cs) == set(t) for t in words)

def wrong_length_only(s, words):       # chỉ so độ dài
    return sum(len(t) <= len(s) for t in words)

wrongs = [
    ("bỏ luật >= 3", wrong_no_three, "heeellooo", ["helo"]),
    ("áp luật >= 3 lên t", wrong_rule_on_t, "aaa", ["a"]),
    ("thiếu kiểm tra hai chuỗi cùng hết", wrong_no_end_check, "abc", ["ab"]),
    ("chỉ đếm ký tự", wrong_counter, "ab", ["ba"]),
    ("chỉ so độ dài", wrong_length_only, "abc", ["xy"]),
]
for name, fn, s, words in wrongs:
    truth = expressive_brute(s, words)
    print(f"{name:36} s={s!r} words={words}: sai = {fn(s, words)}, đúng = {truth}")
    assert fn(s, words) != truth, name
print("12.3 phản ví dụ: OK (các cách sai đều bị lộ)")

bỏ luật >= 3                         s='heeellooo' words=['helo']: sai = 1, đúng = 0
áp luật >= 3 lên t                   s='aaa' words=['a']: sai = 0, đúng = 1
thiếu kiểm tra hai chuỗi cùng hết    s='abc' words=['ab']: sai = 1, đúng = 0
chỉ đếm ký tự                        s='ab' words=['ba']: sai = 1, đúng = 0
chỉ so độ dài                        s='abc' words=['xy']: sai = 1, đúng = 0
12.3 phản ví dụ: OK (các cách sai đều bị lộ)


In [26]:
# 12.4 ngẫu nhiên nhỏ so với oracle BFS (bảng chữ cái nhỏ để có nhiều nhóm trùng ký tự)
rng = random.Random(2809)
n_true = 0
for _ in range(600):
    alpha = rng.choice(["ab", "abc"])
    s = "".join(rng.choice(alpha) for _ in range(rng.randint(1, 11)))
    words = []
    for _ in range(rng.randint(1, 8)):
        if rng.random() < 0.6:
            w = "".join(ch * rng.randint(1, n) for ch, n in rle(s))        # co ngẫu nhiên các nhóm của s
        else:
            w = "".join(rng.choice(alpha) for _ in range(rng.randint(1, 8)))
        words.append(w)
    ref = expressive_brute(s, words)
    n_true += ref
    for name, fn in solutions.items():
        assert fn(s, words) == ref, (name, s, words, fn(s, words), ref)
print(f"12.4 ngẫu nhiên nhỏ so với oracle BFS: OK (tổng số từ stretchy: {n_true})")

# 12.5 ngẫu nhiên lớn tới 100: các cách khớp nhau; chấp nhận khi trùng nhau và bằng tính chất tạo bởi cấu trúc
rng = random.Random(77)
for _ in range(300):
    s = random_runs_string(rng, rng.randint(1, 30))[:100]
    words = []
    for _ in range(rng.randint(1, 100)):
        w = "".join(ch * max(1, min(n, rng.choice([n, 1, 2, 3, n + 1]))) for ch, n in rle(s))[:100]
        words.append(w or "a")
    res = {name: fn(s, words) for name, fn in solutions.items()}
    assert len(set(res.values())) == 1, (s, res)
print("12.5 ngẫu nhiên lớn (độ dài tới 100): OK")

# 12.6 từ sinh bằng cách CO nhóm của s (nhóm >= 3 -> co về bất kỳ >= 1; nhóm < 3 giữ nguyên) luôn stretchy
rng = random.Random(6)
for _ in range(500):
    s = random_runs_string(rng, rng.randint(1, 20), maxrun=8)
    t = "".join(ch * (rng.randint(1, n) if n >= 3 else n) for ch, n in rle(s))
    assert Solution().expressiveWords(s, [t]) == 1, (s, t)
print("12.6 từ sinh bằng cách co nhóm đủ lớn của s luôn stretchy: OK")

12.4 ngẫu nhiên nhỏ so với oracle BFS: OK (tổng số từ stretchy: 1118)


12.5 ngẫu nhiên lớn (độ dài tới 100): OK
12.6 từ sinh bằng cách co nhóm đủ lớn của s luôn stretchy: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Bỏ luật "từ 3 trở lên"

Chỉ cần nhóm của `s` dài hơn hoặc bằng nhóm của `t`? Sai: kéo dài chỉ cho nhóm **>= 3**.

In [27]:
s, words = "heeellooo", ["helo"]
print("sai (c1 >= c2):", wrong_no_three(s, words), "| đúng:", Solution().expressiveWords(s, words))
assert wrong_no_three(s, words) == 1 and Solution().expressiveWords(s, words) == 0

sai (c1 >= c2): 1 | đúng: 0


Vì sao sai: nhóm `ll` (2) trong `s` dài hơn nhóm `l` (1) của `helo`, nhưng 2 < 3 nên không thể sinh bằng một phép kéo dài. **Đúng:** `c1 < c2 or (c1 < 3 and c1 != c2)` -> loại.

### Lỗi 2 — Áp luật `>= 3` lên nhóm của từ truy vấn

Luật nói về kích thước nhóm **sau khi kéo**, tức nhóm của `s`, không phải nhóm của `t`.

In [28]:
s, words = "aaa", ["a"]
print("sai (luật trên t):", wrong_rule_on_t(s, words), "| đúng:", Solution().expressiveWords(s, words))
assert wrong_rule_on_t(s, words) == 0 and Solution().expressiveWords(s, words) == 1

sai (luật trên t): 0 | đúng: 1


Vì sao sai: `"a"` kéo thành `"aaa"` là hợp lệ vì nhóm mới có kích thước 3. **Đúng:** điều kiện trên `c1` (nhóm của `s`).

### Lỗi 3 — Quên kiểm tra cả hai chuỗi cùng hết

Nếu `t` hết sớm (còn nhóm trong `s`) hoặc ngược lại, vòng `while` dừng nhưng chưa chắc khớp.

In [29]:
s, words = "abc", ["ab"]
print("sai (thiếu kiểm tra cuối):", wrong_no_end_check(s, words), "| đúng:", Solution().expressiveWords(s, words))
assert wrong_no_end_check(s, words) == 1 and Solution().expressiveWords(s, words) == 0

sai (thiếu kiểm tra cuối): 1 | đúng: 0


Vì sao sai: sau hai vòng, `t` hết nhưng `s` còn nhóm `c`. **Đúng:** `return i == m and j == n`.

### Lỗi 4 — Chỉ đếm tần suất ký tự

Bỏ qua **thứ tự** và ranh giới nhóm.

In [30]:
s, words = "ab", ["ba"]
print("sai (Counter):", wrong_counter(s, words), "| đúng:", Solution().expressiveWords(s, words))
assert wrong_counter(s, words) == 1 and Solution().expressiveWords(s, words) == 0

sai (Counter): 1 | đúng: 0


Vì sao sai: `"ba"` và `"ab"` có cùng tần suất ký tự nhưng khung khác nhau. **Đúng:** so từng nhóm theo thứ tự.

### Lỗi 5 — Chỉ so độ dài

`len(t) <= len(s)` chỉ là điều kiện cần.

In [31]:
s, words = "abc", ["xy"]
print("sai (độ dài):", wrong_length_only(s, words), "| đúng:", Solution().expressiveWords(s, words))
assert wrong_length_only(s, words) == 1 and Solution().expressiveWords(s, words) == 0

sai (độ dài): 1 | đúng: 0


Vì sao sai: `"xy"` ngắn hơn nhưng ký tự khác hẳn. **Đúng:** so ký tự và độ dài của từng nhóm.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "nhóm ký tự liên tiếp giống nhau", nén/giải nén chuỗi -> **run-length** (`groupby`, hoặc hai con trỏ nhảy theo nhóm).
- **Hai chuỗi so khớp theo cấu trúc nhóm:** luôn kiểm tra (1) khung ký tự, (2) quan hệ độ dài từng cặp nhóm, (3) hai chuỗi cùng hết.
- Luật `>= 3` chỉ phụ thuộc nhóm của `s`: nhóm của `s` có độ dài 1 hoặc 2 buộc nhóm tương ứng của `t` phải **bằng đúng** độ dài đó.
- Khi có **nhiều truy vấn với cùng `s`**, tiền xử lý `s` một lần và dùng khung ký tự để loại nhanh.
- Regex tiện cho việc viết nhanh, nhưng với phỏng vấn nên viết hai con trỏ.
- Mã hóa RLE còn gặp ở: Compress String (443), String Compression II, Decode String, Count and Say (38).
- Test nên có: nhóm 1/2/3 của `s`, từ dài hơn `s`, từ khác khung, từ giống hệt `s`, `s` toàn một ký tự.

## 15. Biến thể

1. **Ngưỡng `k` bất kỳ** thay cho 3: nhóm sau khi kéo phải có kích thước `>= k` (bài tập 5).
2. **Trả về danh sách các từ stretchy** thay vì số lượng.
3. **Cho phép cả co lại** (thu nhỏ nhóm >= 3 về >= 1): cùng khung và mỗi nhóm `>= 1`... (bài khác).
4. Bài liên quan: Compress String, Count and Say, Is Subsequence (392 — cũng so hai chuỗi bằng hai con trỏ).

Dưới đây là code cho biến thể 1 và 2, kiểm bằng oracle BFS có tham số `k`.

In [32]:
def stretchy_k(s: str, t: str, k: int) -> bool:
    rs, rt = rle(s), rle(t)
    if len(rs) != len(rt):
        return False
    for (a, ns), (b, nt) in zip(rs, rt):
        if a != b or ns < nt or (ns < k and ns != nt):
            return False
    return True

def stretchy_words(s: str, words: List[str], k: int = 3) -> List[str]:
    return [t for t in words if stretchy_k(s, t, k)]

assert stretchy_words("heeellooo", ["hello", "hi", "helo"]) == ["hello"]
for s in all_strings("ab", 6):
    for t in all_strings("ab", 5):
        for k in (2, 3, 4):
            ref = len(t) <= len(s) and s in reachable(t, len(s), k)
            assert stretchy_k(s, t, k) == ref, (s, t, k)
print("Biến thể (ngưỡng k): OK")

Biến thể (ngưỡng k): OK


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Sinh mọi chuỗi | Mô phỏng đúng định nghĩa | mũ | rất lớn | Oracle, chuỗi ngắn |
| 2. So RLE | `groupby` + so từng cặp nhóm | `O(m + L)` | `O(m)` | Dễ đọc nhất |
| 3. Hai con trỏ | Nhảy theo nhóm, không tạo danh sách | `O(mW + L)` | `O(1)` | **Cách nộp chính** |
| 4. Regex | Dịch `t` thành mẫu | `O(L)` + khớp | `O(len)` | Viết nhanh; phụ thuộc thư viện |
| 5. Khung chữ cái | Tiền xử lý `s`, loại sớm theo khung | `O(m + L)` | `O(m)` | Nhiều từ, cùng một `s` |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output ba dòng sau, rồi chạy để kiểm tra:

```python
print(rle("aabccc"))
print(Solution().expressiveWords("aaa", ["a", "aa", "aaaa"]))
print(Solution().expressiveWords("aa", ["a", "aa"]))
```

In [33]:
# Bài 1: ghi dự đoán của bạn (chưa chạy code mẫu)
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None

### Bài 2 — Tính tay (dễ-vừa)

Với `s = "abbbcc"`, viết RLE của `s` (list cặp) và với mỗi từ trong `["abc", "abbc", "abbbc", "abbbbcc"]` ghi `True/False` (dưới dạng list 4 phần tử).

In [34]:
# Bài 2: điền kết quả tính tay
hand_rle = None        # [(ký tự, độ dài), ...]
hand_flags = None      # [True/False, ...] cho 4 từ

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành hàm kiểm tra một từ bằng hai con trỏ.

In [35]:
def exercise_3_check(s: str, t: str) -> bool:
    m, n = len(s), len(t)
    i = j = 0
    while i < m and j < n:
        # TODO 1: nếu s[i] != t[j] thì trả False
        # TODO 2: tính c1 = độ dài nhóm bắt đầu tại i trong s
        # TODO 3: tính c2 = độ dài nhóm bắt đầu tại j trong t
        # TODO 4: nếu c1 < c2 hoặc (c1 < 3 và c1 != c2) thì trả False
        # TODO 5: nhảy i += c1, j += c2
        pass
    # TODO 6: trả về True khi cả hai cùng hết
    return None


# assert exercise_3_check("heeellooo", "hello") is True
# assert exercise_3_check("heeellooo", "helo") is False
# assert exercise_3_check("abc", "ab") is False

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 2, hãy tự viết `exercise_4_rle(s, words)` đếm số từ stretchy bằng `groupby`.

In [36]:
def exercise_4_rle(s: str, words: List[str]) -> int:
    # TODO
    pass


# assert exercise_4_rle("heeellooo", ["hello", "hi", "helo"]) == 1
# assert exercise_4_rle("zzzzzyyyyy", ["zzyy", "zy", "zyy"]) == 3
# assert exercise_4_rle("aa", ["a", "aa"]) == 1

### Bài 5 — Mini challenge: ngưỡng `k` bất kỳ (khó)

Phép kéo dài yêu cầu nhóm sau khi kéo có kích thước **`>= k`** (đề gốc `k = 3`). Viết `exercise_5_threshold(s, words, k)` trả số từ stretchy.

```python
exercise_5_threshold("heeellooo", ["hello", "hi", "helo"], 3)   -> 1
exercise_5_threshold("heeellooo", ["hello", "hi", "helo"], 2)   -> 2    (helo: nhóm "ll" = 2 >= 2 nên kéo được)
exercise_5_threshold("aaa", ["a"], 4)                           -> 0
```
Gợi ý: thay hằng số 3 trong điều kiện bằng `k`.

In [37]:
def exercise_5_threshold(s: str, words: List[str], k: int) -> int:
    # TODO
    pass


# assert exercise_5_threshold("heeellooo", ["hello", "hi", "helo"], 3) == 1
# assert exercise_5_threshold("heeellooo", ["hello", "hi", "helo"], 2) == 2
# assert exercise_5_threshold("aaa", ["a"], 4) == 0

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `rle` trả danh sách `(ký tự, độ dài nhóm)`.
- Mức 2: `"aaa"` có một nhóm dài 3; `"aa"` có một nhóm dài 2.
- Mức 3: `[('a', 2), ('b', 1), ('c', 3)]`, `2` (`"a"` và `"aa"` kéo được lên 3; `"aaaa"` dài hơn nên không), `1` (`"aa"` chỉ khớp chính nó vì nhóm 2 < 3).

**Bài 2**
- Mức 1: `"abbbcc"` có 3 nhóm.
- Mức 2: nhóm `bbb` (3) nhận từ có `b` dài 1, 2, 3; nhóm `cc` (2) chỉ nhận đúng `cc`.
- Mức 3: `[('a',1), ('b',3), ('c',2)]`; `abc` -> `c` 1 vs 2 ✗; `abbc` ✗; `abbbc` ✗; `abbbbcc` -> `b` 4 > 3 ✗: tất cả `False` (`[False, False, False, False]`). Thử thêm `abbcc` và `abcc`, sẽ là `True`.

**Bài 3**
- Mức 1: đếm nhóm bằng vòng `while` như mục 3.3.
- Mức 2: điều kiện loại `c1 < c2 or (c1 < 3 and c1 != c2)`.
- Mức 3: cuối hàm `return i == m and j == n`.

**Bài 4**
- Mức 1: `[(ch, len(list(g))) for ch, g in groupby(x)]`.
- Mức 2: hai danh sách phải cùng độ dài; so từng cặp.
- Mức 3: `ok = all(c == d and (ns == nt or (ns >= 3 and ns > nt)) for ...)`.

**Bài 5**
- Mức 1: thay `3` bằng `k` trong điều kiện.
- Mức 2: `ns < nt or (ns < k and ns != nt)` là điều kiện loại.
- Mức 3: kiểm chứng với oracle `reachable(t, len(s), k)`.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [38]:
print(rle("aabccc"))
print(Solution().expressiveWords("aaa", ["a", "aa", "aaaa"]))
print(Solution().expressiveWords("aa", ["a", "aa"]))

assert rle("aabccc") == [("a", 2), ("b", 1), ("c", 3)]
assert Solution().expressiveWords("aaa", ["a", "aa", "aaaa"]) == 2
assert Solution().expressiveWords("aa", ["a", "aa"]) == 1

[('a', 2), ('b', 1), ('c', 3)]
2
1


### Đáp án Bài 2

In [39]:
s = "abbbcc"
hand_rle_solution = [("a", 1), ("b", 3), ("c", 2)]
words = ["abc", "abbc", "abbbc", "abbbbcc"]
flags = [stretchy_brute(s, w) for w in words]
print(flags)
assert rle(s) == hand_rle_solution
assert flags == [False, False, False, False]
assert stretchy_brute(s, "abbcc") and stretchy_brute(s, "abcc") and stretchy_brute(s, "abbbcc")
print("Bài 2 OK")

[False, False, False, False]
Bài 2 OK


### Đáp án Bài 3

In [40]:
def exercise_3_check_solution(s: str, t: str) -> bool:
    m, n = len(s), len(t)
    i = j = 0
    while i < m and j < n:
        if s[i] != t[j]:
            return False
        c1 = 0
        while i + c1 < m and s[i + c1] == s[i]:
            c1 += 1
        c2 = 0
        while j + c2 < n and t[j + c2] == t[j]:
            c2 += 1
        if c1 < c2 or (c1 < 3 and c1 != c2):
            return False
        i += c1
        j += c2
    return i == m and j == n

assert exercise_3_check_solution("heeellooo", "hello") is True
assert exercise_3_check_solution("heeellooo", "helo") is False
assert exercise_3_check_solution("abc", "ab") is False
for s in all_strings("ab", 6):
    for t in all_strings("ab", 5):
        assert exercise_3_check_solution(s, t) == stretchy_two_pointers(s, t), (s, t)
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [41]:
def exercise_4_rle_solution(s: str, words: List[str]) -> int:
    rs = [(ch, len(list(g))) for ch, g in groupby(s)]
    ans = 0
    for t in words:
        rt = [(ch, len(list(g))) for ch, g in groupby(t)]
        if len(rs) == len(rt) and all(c == d and (ns == nt or (ns >= 3 and ns > nt)) for (c, ns), (d, nt) in zip(rs, rt)):
            ans += 1
    return ans

assert exercise_4_rle_solution("heeellooo", ["hello", "hi", "helo"]) == 1
assert exercise_4_rle_solution("zzzzzyyyyy", ["zzyy", "zy", "zyy"]) == 3
assert exercise_4_rle_solution("aa", ["a", "aa"]) == 1
rng = random.Random(4)
for _ in range(300):
    s = "".join(rng.choice("ab") for _ in range(rng.randint(1, 10)))
    words = ["".join(rng.choice("ab") for _ in range(rng.randint(1, 7))) for _ in range(5)]
    assert exercise_4_rle_solution(s, words) == expressive_brute(s, words), (s, words)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [42]:
def exercise_5_threshold_solution(s: str, words: List[str], k: int) -> int:
    rs = rle(s)
    ans = 0
    for t in words:
        rt = rle(t)
        if len(rs) == len(rt) and all(a == b and not (ns < nt or (ns < k and ns != nt)) for (a, ns), (b, nt) in zip(rs, rt)):
            ans += 1
    return ans

assert exercise_5_threshold_solution("heeellooo", ["hello", "hi", "helo"], 3) == 1
assert exercise_5_threshold_solution("heeellooo", ["hello", "hi", "helo"], 2) == 2
assert exercise_5_threshold_solution("aaa", ["a"], 4) == 0
rng = random.Random(5)
for _ in range(300):
    s = "".join(rng.choice("ab") for _ in range(rng.randint(1, 9)))
    words = ["".join(rng.choice("ab") for _ in range(rng.randint(1, 6))) for _ in range(4)]
    k = rng.randint(2, 4)
    ref = sum(len(t) <= len(s) and s in reachable(t, len(s), k) for t in words)
    assert exercise_5_threshold_solution(s, words, k) == ref, (s, words, k)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (hai con trỏ nhảy theo nhóm)

```python
class Solution:
    def expressiveWords(self, s: str, words: List[str]) -> int:
        def check(s, t):
            m, n = len(s), len(t)
            if n > m:
                return False
            i = j = 0
            while i < m and j < n:
                if s[i] != t[j]:
                    return False
                k = i
                while k < m and s[k] == s[i]:
                    k += 1
                c1 = k - i                      # độ dài nhóm trong s
                i, k = k, j
                while k < n and t[k] == t[j]:
                    k += 1
                c2 = k - j                      # độ dài nhóm trong t
                j = k
                if c1 < c2 or (c1 < 3 and c1 != c2):
                    return False
            return i == m and j == n            # cả hai cùng hết

        return sum(check(s, t) for t in words)
```

Công thức nhớ nhanh: mỗi cặp nhóm phải **cùng ký tự** và (`c1 == c2` hoặc (`c1 >= 3` và `c1 > c2`)).

### Checklist trước khi submit

- [ ] So **từng cặp nhóm**, không so tổng độ dài hay tần suất ký tự?
- [ ] Luật `>= 3` áp lên nhóm của **`s`** (`c1`), không phải `t`?
- [ ] Nhóm của `s` dài 1 hoặc 2 thì phải **bằng đúng** nhóm của `t`?
- [ ] Kiểm tra cuối `i == m and j == n` (hai chuỗi cùng hết)?
- [ ] Đã thử `s = "aa"`/`"a"`, từ dài hơn `s`, từ khác khung, `s` toàn một ký tự?
- [ ] Tên method đúng `expressiveWords(self, s, words)` và trả `int`?